# 02 — Evaluate grounding against the GT scene graphs

The GT scene graphs written by `01_record_sweep.ipynb` are the **answer key**. For each
recorded clip we run GroundingDINO on the middle frame under three prompt variants and
score its boxes against `box2d` in the GT graph.

Prompt variants — increasing relational content:

1. `car` — no colour, no relation
2. `red car` — colour only; both red cars are legal
3. `red car behind the bus` — the relational clause

### The two conditions are not symmetric

Both conditions hold the same three vehicles: the bus, a red **target** in the bus's own
lane, and a red **distractor** parked in the opposing lane facing the bus. The distractor
never moves, and is never behind the bus. Only the target moves:

| condition | target | correct answer to *"red car behind the bus"* |
|-----------|--------|----------------------------------------------|
| `behind`  | behind the bus | the **target** |
| `front`   | in front of the bus | **nothing** — no red car is behind the bus |

So `front` is a **negative control**, and `correct_answer_role` is `null` there. A
detector that returns a confident red car in the `front` condition has grounded "red car"
and discarded the relational clause — that is a false positive, and it is the headline
failure this experiment is built to catch.

The flip test still reads the same way: the two conditions are the same scene from the
same camera, so a detector that actually reads "behind the bus" must answer differently
in the two. If its choice never changes, the clause is inert.

Each clip also records `target_relation` (`behind` / `in_front_of`), so the mirror prompt
*"red car in front of the bus"* can be scored off the same images — correct in `front`,
nothing in `behind`.

No tracking, no training. Only the middle frame of each clip is used.


## 0. Imports and manifest

In [ ]:
import os
import json
import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as patches

%matplotlib inline
pd.set_option('display.width', 200)
pd.set_option('display.max_columns', None)   # the aggregate table is 9 columns wide

In [ ]:
SWEEP_ROOT = 'runs/sweep'

with open(os.path.join(SWEEP_ROOT, 'manifest.json')) as f:
    MANIFEST = json.load(f)

CLIPS = MANIFEST['clips']
CONFIG_IDS = sorted({c['config_id'] for c in CLIPS})

print(f'{len(CLIPS)} clips over {len(CONFIG_IDS)} configs '
      f'({len(MANIFEST["skipped"])} configs skipped at record time)')
print('prompt under test:', MANIFEST['prompt'])
print('configs:', ', '.join(CONFIG_IDS))

## 1. Frame loading

Middle frame only: `frames // 2`. Returns the RGB array plus the GT graph for that exact
frame, so detections are always scored against boxes from the same tick.

In [ ]:
def middle_index(clip):
    return clip['frames'] // 2


def load_frame(clip):
    i = middle_index(clip)
    img = np.array(Image.open(os.path.join(clip['rgb_dir'], f'{i:06d}.png')).convert('RGB'))
    with open(os.path.join(clip['gt_dir'], f'{i:06d}.json')) as f:
        graph = json.load(f)
    return img, graph


def gt_boxes_by_role(clip, graph):
    """{'bus': [x1,y1,x2,y2], 'target': ..., 'distractor': ...}"""
    role_of = {v: k for k, v in clip['ids'].items()}
    return {role_of[n['id']]: n['box2d'] for n in graph['nodes'] if n['id'] in role_of}


_img, _graph = load_frame(CLIPS[0])
print(_img.shape, '|', {k: [round(v) for v in b] for k, b in
                        gt_boxes_by_role(CLIPS[0], _graph).items()})

## 2. ⚠️ PLACEHOLDER — your GroundingDINO pipeline goes here

**This is the only cell you need to fill in.** Everything below calls
`run_inference(image, text_prompt)` and nothing else.

Contract:

```
run_inference(image: np.ndarray[H, W, 3] uint8 RGB,
              text_prompt: str)
    -> list[(box, score, phrase)]
       box    : [x1, y1, x2, y2] in ABSOLUTE PIXELS of the input image
       score  : float, higher = more confident
       phrase : str, the text span the box was grounded to
```

If your pipeline returns normalised `cxcywh` (HF `GroundingDinoProcessor` does), convert
it here — the scorer assumes absolute-pixel xyxy.

In [ ]:
# ---------------------------------------------------------------------------
# MODEL INIT -- fill this in
# ---------------------------------------------------------------------------
# e.g.
#   from groundingdino.util.inference import load_model, predict
#   MODEL = load_model('GroundingDINO_SwinT_OGC.py', 'groundingdino_swint_ogc.pth')
#   BOX_THRESHOLD, TEXT_THRESHOLD = 0.25, 0.25

MODEL = None


def run_inference(image, text_prompt):
    """np.ndarray HxWx3 RGB, str -> list of (box_xyxy_pixels, score, phrase)."""
    raise NotImplementedError(
        'Fill in run_inference() with your GroundingDINO pipeline. '
        'Return [x1, y1, x2, y2] in absolute pixels.')

    # sketch, HF style:
    # h, w = image.shape[:2]
    # inputs = PROCESSOR(images=Image.fromarray(image), text=text_prompt, return_tensors='pt')
    # with torch.no_grad():
    #     outputs = MODEL(**inputs)
    # res = PROCESSOR.post_process_grounded_object_detection(
    #     outputs, inputs.input_ids, box_threshold=0.25, text_threshold=0.25,
    #     target_sizes=[(h, w)])[0]
    # return [(b.tolist(), float(s), p)
    #         for b, s, p in zip(res['boxes'], res['scores'], res['labels'])]

### Optional plumbing check

Run this cell **only** to exercise the scoring/aggregation code without a real model —
it replaces `run_inference` with a fake detector that returns jittered GT boxes and
always prefers the same red car regardless of the prompt (i.e. a detector with a
completely inert relational clause). Re-run the cell above to restore the real stub.

In [ ]:
USE_FAKE_DETECTOR = False        # flip to True to smoke-test the pipeline

if USE_FAKE_DETECTOR:
    _fake_rng = np.random.default_rng(0)

    def run_inference(image, text_prompt):
        clip  = _FAKE_CTX['clip']
        boxes = _FAKE_CTX['gt']
        out = []
        for role, base in boxes.items():
            if role == 'bus' and 'bus' not in text_prompt:
                continue
            b = [v + _fake_rng.normal(0, 3) for v in base]
            # always favours whichever red car sits in the 'target' role -> never flips
            s = {'target': 0.9, 'distractor': 0.6, 'bus': 0.5}[role]
            out.append((b, s, role))
        return out

    print('fake detector active -- results are meaningless, plumbing only')

## 3. Scoring

`match_role(box)` returns the role (`target` / `distractor` / `bus`) of the GT box with
the highest IoU against `box`, or `None` if nothing clears 0.5.

| metric | definition |
|--------|------------|
| `selection_correct` | `behind`: role of the **top-scoring** detection == `target`. `front`: the detector picked **no car at all** — the only correct answer there is nothing |
| `false_positive` | `front` only: it picked a car anyway |
| `distractor_confusion` | role of the top-scoring detection == `confuser_role` — the oncoming car in `behind`, the in-front target in `front` |
| `anchor_recall` | **any** detection matches the bus |

Roles, not raw actor ids: the sweep respawns actors per clip so ids differ between the
two conditions of the same config. The manifest records `correct_answer_role` per clip —
`target` in `behind`, `null` in `front`.

`chosen_car_role` is separate: the top-scoring detection that matches *a car* (bus
matches skipped). It is what the flip test compares, so a run where the bus outranks
every car does not silently count as "no flip". It is also what `front` is scored on —
"picked no car" means `chosen_car_role is None`, which a stray box on a building will not
satisfy.


In [ ]:
IOU_THRESH = 0.5
PROMPT_VARIANTS = ['car', 'red car', 'red car behind the bus']


def iou(a, b):
    ix1, iy1 = max(a[0], b[0]), max(a[1], b[1])
    ix2, iy2 = min(a[2], b[2]), min(a[3], b[3])
    iw, ih = max(0.0, ix2 - ix1), max(0.0, iy2 - iy1)
    inter = iw * ih
    if inter <= 0:
        return 0.0
    ua = (a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - inter
    return inter / ua if ua > 0 else 0.0


def match_role(box, gt):
    best_role, best_iou = None, IOU_THRESH
    for role, gbox in gt.items():
        v = iou(box, gbox)
        if v >= best_iou:
            best_role, best_iou = role, v
    return best_role

In [ ]:
def score_detections(dets, gt, clip):
    ranked = sorted(dets, key=lambda d: -d[1])
    roles  = [match_role(b, gt) for b, _, _ in ranked]

    best_role = roles[0] if roles else None
    chosen_car_role = next((r for r in roles if r in ('target', 'distractor')), None)
    correct = clip['correct_answer_role']

    # `correct is None` is the `front` condition: no red car is behind the bus, so the
    # only correct behaviour is to return no car at all. Scored on chosen_car_role rather
    # than best_role, so a stray box on a building does not count as a correct abstention.
    selection_correct = (chosen_car_role is None if correct is None
                         else best_role == correct)

    return {
        'n_det'               : len(ranked),
        'best_role'           : best_role,
        'best_score'          : ranked[0][1] if ranked else np.nan,
        'chosen_car_role'     : chosen_car_role,
        'selection_correct'   : selection_correct,
        'false_positive'      : correct is None and chosen_car_role is not None,
        'distractor_confusion': best_role == clip['confuser_role'],
        'anchor_recall'       : 'bus' in roles,
    }


## 4. Run the sweep evaluation

3 prompt variants x every clip in the manifest, middle frame only. Detections are
cached in `RAW` so the qualitative cell below can redraw without re-running the model.

In [ ]:
_FAKE_CTX = {}
RAW  = {}      # (config_id, condition, prompt) -> list of (box, score, phrase)
rows = []

for clip in CLIPS:
    img, graph = load_frame(clip)
    gt = gt_boxes_by_role(clip, graph)
    _FAKE_CTX.update({'clip': clip, 'gt': gt})

    for prompt in PROMPT_VARIANTS:
        dets = run_inference(img, prompt)
        RAW[(clip['config_id'], clip['condition'], prompt)] = dets

        rows.append({
            'config_id'          : clip['config_id'],
            'condition'          : clip['condition'],
            'prompt'             : prompt,
            'correct_answer_role': clip['correct_answer_role'],
            **score_detections(dets, gt, clip),
        })

    print(f'{clip["config_id"]}/{clip["condition"]}: done')

df = pd.DataFrame(rows)
df['prompt'] = pd.Categorical(df['prompt'], categories=PROMPT_VARIANTS, ordered=True)
print(f'\n{len(df)} rows')
df.head(12)

## 5. Aggregate table

Rows = prompt variant. Columns = the three metrics, split by condition, plus `all`.
Read it as: does adding the relational clause move selection accuracy at all, and does
it move it in *both* conditions or only the one where "red car" alone already wins?

Remember the two columns mean different things. In `behind`, selection accuracy is
"picked the target". In `front` it is "picked no car", because nothing is behind the bus
— so the `front` column of a colour-only prompt like `red car` should be near 0, and the
interesting question is whether `red car behind the bus` lifts it.

The false-positive rate under the table is the same number inverted, for the one
condition where it means something.


In [ ]:
METRICS = ['selection_correct', 'distractor_confusion', 'anchor_recall']
LABELS  = {'selection_correct': 'selection acc',
           'distractor_confusion': 'distractor conf',
           'anchor_recall': 'anchor recall'}

def _block(frame, label):
    m = frame.groupby('prompt', observed=True)[METRICS].mean().reindex(PROMPT_VARIANTS)
    m.columns = pd.MultiIndex.from_product([[label], [LABELS[c] for c in METRICS]])
    return m


table = pd.concat([_block(df[df.condition == 'behind'], 'behind'),
                   _block(df[df.condition == 'front'],  'front'),
                   _block(df,                           'all')], axis=1)

for cond in ('behind', 'front'):
    print(f'n({cond}) = {(df.condition == cond).sum() // len(PROMPT_VARIANTS)} clips per prompt variant')

# `front` only: it returned a red car when none was behind the bus
fp = (df[df.condition == 'front'].groupby('prompt', observed=True)['false_positive']
        .mean().reindex(PROMPT_VARIANTS).round(3))
print('\nfalse-positive rate in `front` (returned a car when the answer was nothing):')
print(fp.to_string())

table.round(3)


## 6. Flip test — the headline number

Per config and prompt: does `chosen_car_role` change between the `behind` and `front`
conditions? Same scene, same camera, same two red cars — only the target has moved.

- **flip rate 1.0** — the detector's answer tracks the relation.
- **flip rate 0.0** — the relational clause is inert; the detector picks the same car
  either way and the phrase "behind the bus" is doing no work.
- **correct-flip rate** — flipped *and* landed right in both: `target` in `behind`, and
  **no car at all** in `front`, since nothing there is behind the bus. A detector that
  flips for the wrong reason — say it swaps to the oncoming distractor rather than
  abstaining — scores on flip rate but not here.


In [ ]:
# (prompt, config_id, condition) -> the car the detector picked, '<none>' if it picked no car
CHOSEN = {(r.prompt, r.config_id, r.condition): (r.chosen_car_role or '<none>')
          for r in df.itertuples()}

pairs = []
for prompt in PROMPT_VARIANTS:
    for cid in CONFIG_IDS:
        b = CHOSEN.get((prompt, cid, 'behind'))
        f = CHOSEN.get((prompt, cid, 'front'))
        if b is None or f is None:
            continue                      # config missing one of the two conditions
        pairs.append({'prompt': prompt, 'config_id': cid, 'behind': b, 'front': f,
                      'flipped': b != f,
                      # correct in `front` is abstention: nothing is behind the bus
                      'correct_flip': (b == 'target' and f == '<none>')})

paired = pd.DataFrame(pairs)
paired['prompt'] = pd.Categorical(paired['prompt'], categories=PROMPT_VARIANTS, ordered=True)

flip = paired.groupby('prompt', observed=True).agg(
    configs=('config_id', 'size'), flipped=('flipped', 'sum'),
    correct_flip=('correct_flip', 'sum'))
flip['flip_rate']         = (flip['flipped'] / flip['configs']).round(3)
flip['correct_flip_rate'] = (flip['correct_flip'] / flip['configs']).round(3)

display(flip)

head = flip.loc['red car behind the bus']
print(f'\nHEADLINE -- prompt "red car behind the bus":')
print(f'  answer changed between conditions on '
      f'{int(head["flipped"])}/{int(head["configs"])} configs (flip rate {head["flip_rate"]})')
print(f'  ...and changed *correctly* on {int(head["correct_flip"])}/{int(head["configs"])} '
      f'(correct-flip rate {head["correct_flip_rate"]})')
if head['flipped'] == 0:
    print('  => relational clause is INERT: the detector ignores "behind the bus".')


In [ ]:
# per-config detail behind the flip numbers
paired[paired.prompt == 'red car behind the bus'].set_index('config_id')[
    ['behind', 'front', 'flipped', 'correct_flip']]

## 7. Qualitative — draw GT and detections on one config

Green = target, orange = distractor, blue = bus (all from the GT graph);
red = detector boxes, labelled with score and grounded phrase.
Set `QUAL_CONFIG` / `QUAL_PROMPT` and re-run — both conditions are drawn side by side
so you can see whether the chosen car moved.

In [ ]:
QUAL_CONFIG = CONFIG_IDS[0]
QUAL_PROMPT = 'red car behind the bus'

ROLE_COLOR = {'target': '#20c020', 'distractor': '#ff9500', 'bus': '#3080ff'}

clips = [c for c in CLIPS if c['config_id'] == QUAL_CONFIG]
fig, axes = plt.subplots(len(clips), 1, figsize=(15, 8 * len(clips)))
axes = np.atleast_1d(axes)

for ax, clip in zip(axes, clips):
    img, graph = load_frame(clip)
    gt = gt_boxes_by_role(clip, graph)
    ax.imshow(img)

    for role, (x1, y1, x2, y2) in gt.items():
        ax.add_patch(patches.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False,
                                       edgecolor=ROLE_COLOR[role], linewidth=3))
        ax.text(x1, y1 - 8, role, color=ROLE_COLOR[role], fontsize=11, weight='bold')

    for box, score, phrase in RAW.get((clip['config_id'], clip['condition'], QUAL_PROMPT), []):
        x1, y1, x2, y2 = box
        ax.add_patch(patches.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False,
                                       edgecolor='red', linewidth=2, linestyle='--'))
        ax.text(x1, y2 + 20, f'{phrase} {score:.2f}', color='red', fontsize=10)

    r = df[(df.config_id == clip['config_id']) & (df.condition == clip['condition'])
           & (df.prompt == QUAL_PROMPT)].iloc[0]
    correct = clip['correct_answer_role'] or 'nothing (negative control)'
    ax.set_title(f'{clip["config_id"]} / {clip["condition"]}   "{QUAL_PROMPT}"   '
                 f'correct={correct}  '
                 f'chose={r["chosen_car_role"] or "no car"}  '
                 f'{"HIT" if r["selection_correct"] else "MISS"}')
    ax.axis('off')

plt.tight_layout()
plt.show()

## 8. Save the scored table

Written next to the sweep so a later session can reload the numbers without re-running
the detector.

In [ ]:
out_csv = os.path.join(SWEEP_ROOT, 'eval_grounding.csv')
df.to_csv(out_csv, index=False)
flip.to_csv(os.path.join(SWEEP_ROOT, 'eval_flip_test.csv'))
print('wrote', out_csv, 'and eval_flip_test.csv')